# 02. Model Training & Experimentation

**Th?nh vi?n ph? tr?ch:** Th?nh vi?n 2 - Model Development & Training

**M?c ti?u:**
1. N?p d? li?u NBA ?? x? l? (`train.csv`, `val.csv`, `test.csv`) t? `data/processed/`.
2. Kh?i t?o 4 m? h?nh h?c m?y:
   - `DummyClassifier(strategy='prior')`: baseline
   - `LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)`
   - `RandomForestClassifier(class_weight='balanced', random_state=42)`
   - `HistGradientBoostingClassifier(random_state=42)`
3. Hu?n luy?n c?c m? h?nh tr?n t?p Train (m?a gi?i $\le 2016$).
4. D? b?o x?c su?t (y_proba) v? xu?t file `reports/predictions.csv` c? ??ng 6 c?t:
   `['player', 'season', 'split', 'model', 'y_true', 'y_proba']`.
5. L?u tr? c?c m? h?nh ?? fit v?o th? m?c `models/` b?ng `joblib`.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
from src.train import (
    get_default_models,
    train_models,
    export_predictions,
    save_models
)

print("Imports th?nh c?ng t? src.train!")

## B??c 1: N?p d? li?u t? `data/processed/`

In [ ]:
processed_dir = ROOT_DIR / "data" / "processed"

train_df = pd.read_csv(processed_dir / "train.csv")
val_df = pd.read_csv(processed_dir / "val.csv")
test_df = pd.read_csv(processed_dir / "test.csv")

target_col = "All_Star" if "All_Star" in train_df.columns else "is_all_star"

feature_cols = [
    'pts_per_game_zscore', 'ast_per_game_zscore', 'trb_per_game_zscore',
    'per_zscore', 'ws_zscore', 'vorp_zscore',
    'team_win_pct', 'prev_allstar_count',
    'mp_per_game', 'pts_per_game', 'ast_per_game', 'trb_per_game',
    'stl_per_game', 'blk_per_game', 'tov_per_game',
    'fg_percent', 'x3p_percent', 'ft_percent',
    'per', 'ts_percent', 'usg_percent', 'ws', 'bpm', 'vorp'
]
feature_cols = [c for c in feature_cols if c in train_df.columns]

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_val = val_df[feature_cols]
y_val = val_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

print(f"S? l??ng ??c tr?ng s? d?ng ({len(feature_cols)}): {feature_cols}")
print(f"K?ch th??c t?p d? li?u: Train={train_df.shape}, Val={val_df.shape}, Test={test_df.shape}")
print("Ph?n b? nh?n All_Star tr?n t?p Train:")
print(y_train.value_counts(normalize=True))

## B??c 2: Kh?i t?o 4 m? h?nh ph?n lo?i (`get_default_models`)

In [ ]:
models = get_default_models(random_state=42)
for name, m in models.items():
    print(f"- {name}: {m}")

## B??c 3: Hu?n luy?n c?c m? h?nh tr?n t?p Train (`train_models`)

In [ ]:
fitted_models = train_models(models, X_train, y_train)
print("Ho?n t?t hu?n luy?n 4 m? h?nh!")

## B??c 4: Xu?t k?t qu? d? ?o?n ra `reports/predictions.csv` (`export_predictions`)
File CSV xu?t ra c? ??ng 6 c?t: `['player', 'season', 'split', 'model', 'y_true', 'y_proba']`.

In [ ]:
reports_dir = ROOT_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
predictions_path = reports_dir / "predictions.csv"

# D? ?o?n cho t?p validation (2017-2019)
val_preds = export_predictions(
    models_dict=fitted_models,
    X_df=val_df,
    y_true=y_val,
    split_name="val",
    feature_cols=feature_cols
)

# D? ?o?n cho t?p test (2020+)
test_preds = export_predictions(
    models_dict=fitted_models,
    X_df=test_df,
    y_true=y_test,
    split_name="test",
    feature_cols=feature_cols
)

# H?p nh?t v? xu?t CSV
all_preds = pd.concat([val_preds, test_preds], ignore_index=True)
all_preds.to_csv(predictions_path, index=False)

print(f"?? xu?t file d? ?o?n ({len(all_preds)} d?ng) v?o: {predictions_path}")
print("6 c?t chu?n:", list(all_preds.columns))
all_preds.head(10)

## B??c 5: L?u tr? m? h?nh v?o th? m?c `models/` (`save_models`)

In [ ]:
models_dir = ROOT_DIR / "models"
saved_paths = save_models(fitted_models, models_dir)
for name, p in saved_paths.items():
    print(f"?? l?u m? h?nh: {name} -> {p.name}")